# CocoOptima – Component 1: Factory-Level Batch Yield Prediction
**Model training, evaluation, packaging and manual testing**

| Item | Value |
|---|---|
| Target | `saleable_yield_pct` = accepted final product ÷ product-specific feed × 100 (SRS §5.4) |
| Model routing | 4 product-specific models: Desiccated Coconut · Coconut Milk · VCO – Cold Press · VCO – Wet Centrifuge (YLD-FR-06) |
| Baselines | Historical average · 30-day moving average · Multiple linear regression (SRS §5.5) |
| Candidates | Ridge · Random Forest · Extra Trees · Hist Gradient Boosting · SVR · XGBoost · LightGBM · CatBoost (when installed) + weighted ensemble of the best 4 |
| Validation | Chronological: train 2016–2022 · validation 2023–2024 (model selection) · test 2025 (evaluated once) |
| Outputs | Predicted yield %, 80/90 % prediction interval, expected accepted output, High/Moderate/Low level, top factors, warnings |

**How to run:** open this notebook from `01-yield-prediction/notebooks/` and *Run All*.
Training artifacts are written to `01-yield-prediction/model_artifacts/`. Section 9 is the **manual testing** area – edit the input values and run the cell.

`N_ITER` (random-search iterations per model) controls accuracy vs. run time: 30 is the default; raise to 60 for an extra search.

## 1. Setup

In [1]:
import os, sys, json, time, math, re, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import sklearn
from scipy.stats import randint, uniform, loguniform
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.ensemble import (RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor, VotingRegressor,
                              RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier)
from sklearn.svm import SVR
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, f1_score, precision_score, recall_score,
                             accuracy_score, confusion_matrix, roc_auc_score)
from sklearn.inspection import permutation_importance
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

FAST_MODE = os.environ.get("COCO_FAST", "0") == "1"   # quick smoke test only; leave False for real training
RANDOM_STATE = 42
N_ITER = 3 if FAST_MODE else 30                         # random-search iterations per candidate model
N_JOBS = -1
np.random.seed(RANDOM_STATE)


def find_component_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data" / "processed").exists() and (q / "data_generation").exists():
            return q
    raise FileNotFoundError("Open this notebook from inside its component folder (.../notebooks/).")


ROOT = find_component_root()
DATA = ROOT / "data"
ART = ROOT / "model_artifacts"
ART.mkdir(exist_ok=True)

try:
    from IPython.display import display
except Exception:
    def display(x):
        print(x.to_string() if hasattr(x, "to_string") else x)


def optional_import(name):
    try:
        return __import__(name)
    except Exception:
        return None


xgb = optional_import("xgboost")
lgb = optional_import("lightgbm")
cb = optional_import("catboost")
widgets = optional_import("ipywidgets")
try:
    import matplotlib
    import matplotlib.pyplot as plt
except Exception:
    plt = None

LIB_VERSIONS = {"python": sys.version.split()[0], "numpy": np.__version__, "pandas": pd.__version__, "scikit-learn": sklearn.__version__,
                "xgboost": getattr(xgb, "__version__", None), "lightgbm": getattr(lgb, "__version__", None),
                "catboost": getattr(cb, "__version__", None)}
print("Component folder :", ROOT)
print("Libraries        :", LIB_VERSIONS)
print("Optional         : xgboost", bool(xgb), "| lightgbm", bool(lgb), "| catboost", bool(cb), "| ipywidgets", bool(widgets),
      "| matplotlib", bool(plt))
print("FAST_MODE =", FAST_MODE, "| N_ITER =", N_ITER)

Component folder : E:\Coco Optima\ml-components\01-yield-prediction
Libraries        : {'python': '3.11.0', 'numpy': '2.4.6', 'pandas': '3.0.6', 'scikit-learn': '1.9.1', 'xgboost': None, 'lightgbm': None, 'catboost': None}
Optional         : xgboost False | lightgbm False | catboost False | ipywidgets False | matplotlib False
FAST_MODE = False | N_ITER = 30


## 2. Load data

In [2]:
df = pd.read_csv(DATA / "processed" / "yield_all_products.csv", parse_dates=["production_date"])
dd = pd.read_csv(DATA / "data_dictionary" / "data_dictionary.csv")
THRESHOLDS = json.load(open(DATA / "splits" / "yield_thresholds_v1.json"))
print(f"Rows: {len(df):,} | columns: {df.shape[1]} | {df.production_date.min():%Y-%m-%d} -> {df.production_date.max():%Y-%m-%d}")
print("Duplicate batch_id:", int(df.batch_id.duplicated().sum()))
display(df.groupby(["model_group", "split"]).size().unstack().fillna(0).astype(int))
display(df.groupby("model_group").qc_status.value_counts(normalize=True).unstack().round(4))

Rows: 20,765 | columns: 86 | 2016-01-01 -> 2025-12-31
Duplicate batch_id: 0


split,test,train,validation
model_group,,,
Coconut Milk,611,4403,1304
Desiccated Coconut,806,6260,1946
VCO - Cold Press,455,3175,937
VCO - Wet Centrifuge,192,252,424


qc_status,Partial reject,Pass,Rejected/Downgraded
model_group,,,
Coconut Milk,0.0639,0.9313,0.0047
Desiccated Coconut,0.0498,0.9440,0.0062
VCO - Cold Press,0.0007,0.9851,0.0142
VCO - Wet Centrifuge,NaN,0.9988,0.0012


## 3. Exploratory checks

In [3]:
summary = df.groupby(["model_group", "product_variant"]).saleable_yield_pct.describe()[["count", "mean", "std", "min", "50%", "max"]].round(2)
display(summary)
display(df.pivot_table(index=df.production_date.dt.year, columns="model_group", values="saleable_yield_pct", aggfunc="median").round(2))
if plt is not None:
    fig, axes = plt.subplots(1, 4, figsize=(18, 3.5))
    for ax, (g, sub) in zip(axes, df.groupby("model_group")):
        ax.hist(sub.saleable_yield_pct.clip(sub.saleable_yield_pct.quantile(0.01), None), bins=50, color="#2f6f9f")
        ax.set_title(g, fontsize=10); ax.set_xlabel("saleable yield %")
    plt.tight_layout(); plt.show()

count    mean   std    min     50%     max
model_group          product_variant                                                
Coconut Milk         Coconut Cream 24%   1778.0   97.50  8.06  13.51   98.38  111.79
                     Coconut Milk 17%    2987.0  147.70  9.26  21.30  148.61  167.88
                     Coconut Milk 20%    1553.0  122.97  7.51  33.03  123.87  138.62
Desiccated Coconut   Chips               1218.0   47.70  3.22   1.76   48.09   51.81
                     Extra Fine           811.0   45.88  3.58   0.71   46.27   49.99
                     Fine                3994.0   47.27  3.27   1.47   47.66   51.40
                     Medium              2989.0   46.71  3.59   0.74   47.15   51.39
VCO - Cold Press     Virgin Coconut Oil  4567.0   46.58  5.75   0.00   47.38   53.01
VCO - Wet Centrifuge Virgin Coconut Oil   868.0   20.33  0.97   8.09   20.35   22.75

model_group,Coconut Milk,Desiccated Coconut,VCO - Cold Press,VCO - Wet Centrifuge
production_date,,,,
2016,129.23,47.24,47.55,NaN
2017,125.96,47.62,46.17,NaN
2018,130.23,47.72,47.66,NaN
2019,130.86,47.63,47.90,NaN
2020,127.06,47.57,47.07,NaN
2021,129.49,47.68,47.90,20.34
2022,128.90,46.88,47.47,20.62
2023,128.43,47.67,47.68,20.66
2024,126.22,47.07,47.31,20.10


## 4. Feature engineering module
The feature code is written to `model_artifacts/c1_features.py` and used **inside** the saved pipeline (`FunctionTransformer`),
so the backend applies exactly the same transformations (SRS §3.3 – no manual re-creation of transforms).

In [4]:
FEATURE_MODULE = r"""
# Component 1 feature engineering - shared by training and inference (imported by the saved pipelines).
import numpy as np
import pandas as pd

CATEGORICAL = ["product_variant", "line_id", "machine_id", "shift", "crew_team", "source_region", "blanching_method",
               "pack_format", "extraction_method", "event_context", "monsoon_season", "batch_size_category"]
DRYER_WATER_CAPACITY_KG = {"DRY-A": 1650.0, "DRY-C": 1450.0, "DRY-B": 1900.0}
MAINT_INTERVAL_DAYS = {"DRY-A": 35, "DRY-C": 30, "DRY-B": 40, "PRS-1": 28, "PRS-2": 28, "EXP-1": 21, "EXP-2": 21, "CFG-1": 30}
ENGINEERED = ["lab_kernel_fat_pct_db", "feed_fat_pct_est", "fat_input_kg", "dryer_load_index", "dryer_overload", "wear_ratio_sq",
              "fat_to_target_ratio", "dk_moisture_dev_sq", "night_or_evening_shift", "heavy_rain_week", "nut_price_per_mn_nuts",
              "lab_moisture_x_immature"]


def _num(d, c):
    if c in d.columns:
        return pd.to_numeric(d[c], errors="coerce").astype(float)
    return pd.Series(np.nan, index=d.index, dtype=float)


def add_features(df):
    d = df.copy()
    for c in d.columns:
        if c in CATEGORICAL:
            d[c] = d[c].astype(object).where(d[c].notna(), "Unknown").astype(str)
        elif d[c].dtype == object or str(d[c].dtype) in ("string", "str"):
            conv = pd.to_numeric(d[c], errors="coerce")
            if conv.notna().sum() >= d[c].notna().sum():
                d[c] = conv
    for c in CATEGORICAL:
        if c not in d.columns:
            d[c] = "Unknown"
    machine = d["machine_id"].astype(str)
    feed, moist = _num(d, "product_specific_feed_kg"), _num(d, "lab_kernel_moisture_pct")
    cap = machine.map(DRYER_WATER_CAPACITY_KG).astype(float)
    d["dryer_load_index"] = feed * (moist + 1.2 - 2.5) / 100.0 / cap
    d["dryer_overload"] = (d["dryer_load_index"] - 1.0).clip(lower=0)
    interval = machine.map(MAINT_INTERVAL_DAYS).astype(float)
    d["wear_ratio_sq"] = (_num(d, "days_since_maintenance") / interval) ** 2
    d["fat_to_target_ratio"] = _num(d, "lab_kernel_fat_pct_wb") / _num(d, "target_fat_pct")
    d["dk_moisture_dev_sq"] = (_num(d, "feed_moisture_pct") - 7.2) ** 2
    d["night_or_evening_shift"] = d["shift"].isin(["Night", "Evening"]).astype(int)
    d["heavy_rain_week"] = (_num(d, "rain_7d_mm") > 100).astype(int)
    d["nut_price_per_mn_nuts"] = _num(d, "retail_nut_price_lkr_lag1m") / _num(d, "national_coconut_prod_mn_nuts_lag2m")
    fat_wb = _num(d, "lab_kernel_fat_pct_wb")
    d["lab_kernel_fat_pct_db"] = fat_wb / (1 - moist / 100.0)
    fm = _num(d, "feed_moisture_pct")
    d["feed_fat_pct_est"] = np.where(fm.notna(), d["lab_kernel_fat_pct_db"] * (1 - fm / 100.0), fat_wb)
    d["fat_input_kg"] = feed * d["feed_fat_pct_est"] / 100.0
    d["lab_moisture_x_immature"] = moist * _num(d, "immature_nut_pct")
    return d
"""
(ART / "c1_features.py").write_text(FEATURE_MODULE.strip() + "\n", encoding="utf-8")
if str(ART) not in sys.path:
    sys.path.insert(0, str(ART))
import importlib
import c1_features
importlib.reload(c1_features)
print("Feature module written:", ART / "c1_features.py")
display(c1_features.add_features(df.head(3))[c1_features.ENGINEERED])

Feature module written: E:\Coco Optima\ml-components\01-yield-prediction\model_artifacts\c1_features.py


,lab_kernel_fat_pct_db,feed_fat_pct_est,fat_input_kg,dryer_load_index,dryer_overload,wear_ratio_sq,fat_to_target_ratio,dk_moisture_dev_sq,night_or_evening_shift,heavy_rain_week,nut_price_per_mn_nuts,lab_moisture_x_immature
0,67.044123,34.34,719.49168,NaN,NaN,0.154337,2.020,NaN,0,0,0.158326,174.6324
1,67.044123,34.34,904.75598,NaN,NaN,0.154337,1.717,NaN,0,0,0.158326,174.6324
2,67.044123,34.34,747.27274,NaN,NaN,0.154337,2.020,NaN,1,0,0.158326,174.6324


## 5. Feature sets per model group
Only fields that the data dictionary marks as **pre-production predictors** are used (leakage prevention, SRS §10.4).
Outcomes (accepted/rejected output, QC status, final quality) and in-process checkpoint fields are excluded from the initial-prediction models.

In [5]:
GROUPS = {"Desiccated Coconut": "dc", "Coconut Milk": "cm", "VCO - Cold Press": "vco_cold_press", "VCO - Wet Centrifuge": "vco_wet_centrifuge"}
# derived/duplicate fields are recomputed inside c1_features.add_features (consistent for manual inputs)
DROP_ALWAYS = {"year", "public_holiday", "feed_description", "dried_kernel_kg", "lab_kernel_fat_pct_db", "feed_fat_input_kg",
               "lab_fat_missing", "nuts_allocated", "whole_kernel_kg", "paring_loss_kg", "pared_kernel_kg", "kernel_recovery_pct",
               "batch_size_category", "days_since_lot_receipt", "quarter", "dk_drying_temp_c", "dk_drying_hours"}
PRED_FIELDS = dd[(dd.allowed_as_predictor == "yes") & (dd.availability == "pre-production")].field.tolist()
REQUIRED_BASE = ["product_specific_feed_kg", "lab_kernel_moisture_pct", "machine_id", "shift", "production_date"]
REQUIRED_EXTRA = {"Desiccated Coconut": ["product_variant", "planned_dryer_inlet_temp_c", "planned_dryer_residence_min"],
                  "Coconut Milk": ["product_variant", "target_fat_pct", "planned_water_kernel_ratio"],
                  "VCO - Cold Press": ["feed_moisture_pct", "planned_feed_rate_kg_h", "planned_expeller_passes"],
                  "VCO - Wet Centrifuge": []}
TARGET = "saleable_yield_pct"

FEATURES = {}
for g in GROUPS:
    tr = df[(df.model_group == g) & (df.split == "train")]
    raw = [c for c in PRED_FIELDS if c in df.columns and c not in DROP_ALWAYS and tr[c].notna().any() and tr[c].nunique(dropna=True) > 1]
    eng = c1_features.add_features(tr[raw])
    eng_cols = [c for c in c1_features.ENGINEERED if eng[c].notna().any() and eng[c].nunique(dropna=True) > 1]
    cat = [c for c in raw if c in c1_features.CATEGORICAL]
    num = [c for c in raw if c not in cat] + eng_cols
    req = [c for c in REQUIRED_BASE + REQUIRED_EXTRA[g] if c in raw]
    FEATURES[g] = {"raw": raw, "num": num, "cat": cat, "engineered": eng_cols, "required": req}
    print(f"{g:22s} raw inputs={len(raw):2d}  numeric(model)={len(num):2d}  categorical={len(cat):2d}  engineered={len(eng_cols)}")

Desiccated Coconut     raw inputs=36  numeric(model)=37  categorical= 9  engineered=10
Coconut Milk           raw inputs=37  numeric(model)=37  categorical= 9  engineered=9
VCO - Cold Press       raw inputs=37  numeric(model)=39  categorical= 6  engineered=8
VCO - Wet Centrifuge   raw inputs=31  numeric(model)=32  categorical= 5  engineered=6


## 6. Modelling utilities

In [6]:
def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", s.lower()).strip("_")


def make_preprocessor(num, cat, scale=False):
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    try:
        ohe = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=5, sparse_output=False)
    except TypeError:                                   # very old scikit-learn
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    return ColumnTransformer([("num", Pipeline(num_steps), num), ("cat", ohe, cat)], remainder="drop")


def reg_metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": round(float(mean_absolute_error(y, p)), 4), "RMSE": round(float(np.sqrt(mean_squared_error(y, p))), 4),
            "R2": round(float(r2_score(y, p)), 4)}


def _trees(lo, hi):
    return randint(40, 80) if FAST_MODE else randint(lo, hi)


def regressor_candidates(n_rows):
    c = {
        "Ridge regression": (Ridge(), {"model__alpha": loguniform(1e-3, 300)}, True),
        "Random forest": (RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
                          {"model__n_estimators": _trees(300, 800), "model__max_depth": [None, 10, 16, 24],
                           "model__min_samples_leaf": randint(1, 8), "model__max_features": [0.3, 0.5, 0.7, 1.0]}, False),
        "Extra trees": (ExtraTreesRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS),
                        {"model__n_estimators": _trees(300, 800), "model__max_depth": [None, 12, 20, 30],
                         "model__min_samples_leaf": randint(1, 6), "model__max_features": [0.4, 0.6, 0.8, 1.0]}, False),
        "Hist gradient boosting": (HistGradientBoostingRegressor(random_state=RANDOM_STATE),
                                   {"model__learning_rate": loguniform(0.02, 0.2), "model__max_iter": _trees(200, 900),
                                    "model__max_leaf_nodes": randint(8, 64), "model__min_samples_leaf": randint(5, 60),
                                    "model__l2_regularization": loguniform(1e-3, 10)}, False),
    }
    if n_rows <= 9000:
        c["Support vector regression"] = (SVR(), {"model__C": loguniform(0.3, 300), "model__epsilon": loguniform(0.005, 1.0),
                                                  "model__gamma": ["scale", 0.003, 0.01, 0.03]}, True)
    if xgb is not None:
        c["XGBoost"] = (xgb.XGBRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, tree_method="hist", verbosity=0),
                        {"model__n_estimators": _trees(300, 1500), "model__learning_rate": loguniform(0.01, 0.2),
                         "model__max_depth": randint(2, 9), "model__subsample": uniform(0.6, 0.4),
                         "model__colsample_bytree": uniform(0.5, 0.5), "model__min_child_weight": randint(1, 12),
                         "model__reg_lambda": loguniform(0.1, 20)}, False)
    if lgb is not None:
        c["LightGBM"] = (lgb.LGBMRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS, verbose=-1, subsample_freq=1),
                         {"model__n_estimators": _trees(300, 1500), "model__learning_rate": loguniform(0.01, 0.2),
                          "model__num_leaves": randint(8, 96), "model__min_child_samples": randint(5, 60),
                          "model__subsample": uniform(0.6, 0.4), "model__colsample_bytree": uniform(0.5, 0.5),
                          "model__reg_lambda": loguniform(1e-3, 20)}, False)
    if cb is not None:
        c["CatBoost"] = (cb.CatBoostRegressor(random_seed=RANDOM_STATE, verbose=0, allow_writing_files=False, thread_count=-1),
                         {"model__iterations": _trees(300, 1500), "model__learning_rate": loguniform(0.02, 0.2),
                          "model__depth": randint(4, 9), "model__l2_leaf_reg": loguniform(1, 20)}, False)
    return c


from sklearn.model_selection import ParameterSampler
from sklearn.metrics import get_scorer

PROGRESS_LOG = ART / "training_progress.log"
PLAN = {"total": 0, "done": 0, "t0": time.time()}


def fmt_time(sec):
    sec = int(max(sec, 0))
    return f"{sec // 3600}:{sec % 3600 // 60:02d}:{sec % 60:02d}"


def log(msg):
    """Print a time-stamped progress line and append it to model_artifacts/training_progress.log."""
    line = f"{time.strftime('%H:%M:%S')} | {msg}"
    print(line, flush=True)
    with open(PROGRESS_LOG, "a", encoding="utf-8") as fh:
        fh.write(line + "\n")


def plan_models(n, title):
    """Declare how many models will be tuned in the next stage (used for the overall ETA)."""
    PLAN.update({"total": PLAN["done"] + int(n), "t0": PLAN["t0"] if PLAN["done"] else time.time()})
    log(f"==== {title}: {n} models to tune x {N_ITER} trials each ====")


METRIC_NAME = {"neg_mean_absolute_error": "MAE", "roc_auc": "AUC", "f1_macro": "macro-F1"}


def tune_on_validation(pipe, space, Xtr, ytr, Xva, yva, scoring="neg_mean_absolute_error", label="model"):
    """Random search: each trial trains on the training period and is scored on the validation period.
    Prints live progress (trial i/N, score, best so far, time per trial, elapsed, ETA)."""
    scorer = get_scorer(scoring)
    sign = -1 if scoring.startswith("neg_") else 1
    mname = METRIC_NAME.get(scoring, scoring)
    params = {}
    t_model = time.time()
    if space:
        trials = list(ParameterSampler(space, n_iter=N_ITER, random_state=RANDOM_STATE))
        best = None
        for i, p in enumerate(trials, 1):
            try:
                m = clone(pipe).set_params(**p).fit(Xtr, ytr)
                sc = float(scorer(m, Xva, yva))
            except Exception as e:
                sc = np.nan
                log(f"   {label} trial {i}: failed ({type(e).__name__}: {str(e)[:80]})")
            if np.isfinite(sc) and (best is None or sc > best[0]):
                best = (sc, p)
            el = time.time() - t_model
            per = el / i
            shown = "   n/a" if not np.isfinite(sc) else f"{sign * sc:.4f}"
            bshown = "   n/a" if best is None else f"{sign * best[0]:.4f}"
            log(f"   {label:30s} trial {i:>3}/{len(trials)} | {mname} {shown} | best {bshown} | {per:5.1f}s/trial | "
                f"elapsed {fmt_time(el)} | ETA {fmt_time(per * (len(trials) - i))}")
        if best is None:
            raise RuntimeError("all trials failed")
        params = best[1]
        pipe.set_params(**params)
    pipe.fit(Xtr, ytr)
    PLAN["done"] += 1
    el_all = time.time() - PLAN["t0"]
    total = max(PLAN["total"], PLAN["done"])
    eta_all = el_all / PLAN["done"] * (total - PLAN["done"])
    log(f"## {label} finished in {fmt_time(time.time() - t_model)} | overall model {PLAN['done']}/{total} "
        f"({PLAN['done'] / total * 100:.0f} %) | notebook elapsed {fmt_time(el_all)} | estimated remaining {fmt_time(eta_all)}")
    return pipe, params

In [7]:
def split_xy(g, splits, include_downgraded=False):
    m = (df.model_group == g) & (df.split.isin(splits))
    if not include_downgraded:
        m &= df.qc_status != "Rejected/Downgraded"
    sub = df[m].sort_values("production_date")
    return sub, sub[FEATURES[g]["raw"]], sub[TARGET].values


def yield_level(g, variant, y):
    t = THRESHOLDS["thresholds"].get(f"{g} | {variant}") or next(v for k, v in THRESHOLDS["thresholds"].items() if k.startswith(g))
    return "Low" if y < t["low_below"] else ("High" if y >= t["high_at_or_above"] else "Moderate")


def make_pipeline(g, estimator, scale):
    f = FEATURES[g]
    return Pipeline([("features", FunctionTransformer(c1_features.add_features)),
                     ("prep", make_preprocessor(f["num"], f["cat"], scale)),
                     ("model", estimator)])

## 7. Baselines and tuned candidate models (selection on validation 2023–2024)
Each candidate is tuned with a random search that trains on 2016–2022 and scores on 2023–2024 (time-ordered, no shuffling).
Batches fully rejected/downgraded by QC (≈1 %, random microbiological events) are excluded from training and reported separately.

In [8]:
from scipy.optimize import nnls
RESULTS, FITTED, BEST_PARAMS = {}, {}, {}
plan_models(sum(len(regressor_candidates(len(split_xy(g, ["train"])[2]))) for g in GROUPS), "Component 1 - yield models for 4 product groups")
for gi, g in enumerate(GROUPS, 1):
    t_g = time.time()
    log(f">>> product group {gi}/{len(GROUPS)}: {g}")
    _, Xtr, ytr = split_xy(g, ["train"])
    va_df, Xva, yva = split_xy(g, ["validation"])
    rows = []
    # --- baselines
    tr_df = df[(df.model_group == g) & (df.split == "train") & (df.qc_status != "Rejected/Downgraded")]
    var_mean = tr_df.groupby("product_variant")[TARGET].mean()
    p_hist = va_df.product_variant.map(var_mean).fillna(ytr.mean()).values
    p_ma = va_df.hist_yield_mean_30d_variant.fillna(va_df.product_variant.map(var_mean)).fillna(ytr.mean()).values
    rows.append({"model": "Baseline: historical average", **reg_metrics(yva, p_hist)})
    rows.append({"model": "Baseline: 30-day moving average", **reg_metrics(yva, p_ma)})
    mlr = make_pipeline(g, LinearRegression(), True).fit(Xtr, ytr)
    rows.append({"model": "Baseline: multiple linear regression", **reg_metrics(yva, mlr.predict(Xva))})
    # --- tuned candidates
    FITTED[g], BEST_PARAMS[g] = {}, {}
    for name, (est, space, scale) in regressor_candidates(len(Xtr)).items():
        t0 = time.time()
        try:
            pipe, params = tune_on_validation(make_pipeline(g, est, scale), space, Xtr, ytr, Xva, yva, label=f"{GROUPS[g]} | {name}")
        except Exception as e:
            print(f"  ! {name} skipped: {type(e).__name__}: {e}")
            continue
        FITTED[g][name], BEST_PARAMS[g][name] = pipe, params
        rows.append({"model": name, **reg_metrics(yva, pipe.predict(Xva)), "fit_s": round(time.time() - t0, 1)})
    # --- weighted ensemble of the 4 best candidates (non-negative weights fitted on validation predictions)
    cand = [r["model"] for r in sorted([r for r in rows if r["model"] in FITTED[g]], key=lambda r: r["MAE"])[:4]]
    if len(cand) >= 2:
        P = np.column_stack([FITTED[g][m].predict(Xva) for m in cand])
        w, _ = nnls(P, yva)
        w = w / w.sum() if w.sum() > 0 else np.full(len(cand), 1 / len(cand))
        keep = [i for i in range(len(cand)) if w[i] > 0.02]
        if len(keep) >= 2:
            ens = VotingRegressor([(slug(cand[i]), clone(FITTED[g][cand[i]])) for i in keep],
                                  weights=[float(w[i] / w[keep].sum()) for i in keep]).fit(Xtr, ytr)
            name = "Weighted ensemble (" + ", ".join(f"{cand[i]} {w[i] / w[keep].sum():.2f}" for i in keep) + ")"
            FITTED[g][name] = ens
            rows.append({"model": name, **reg_metrics(yva, ens.predict(Xva))})
    RESULTS[g] = pd.DataFrame(rows).set_index("model").sort_values("MAE")
    print(f"\n=== {g}  (validation 2023-2024, n={len(yva)}, {time.time() - t_g:.0f}s)")
    display(RESULTS[g])

07:36:16 | ==== Component 1 - yield models for 4 product groups: 20 models to tune x 30 trials each ====
07:36:16 | >>> product group 1/4: Desiccated Coconut
07:36:17 |    dc | Ridge regression          trial   1/30 | MAE 0.4621 | best 0.4621 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:36:17 |    dc | Ridge regression          trial   2/30 | MAE 0.4659 | best 0.4621 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:36:17 |    dc | Ridge regression          trial   3/30 | MAE 0.4592 | best 0.4592 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:36:17 |    dc | Ridge regression          trial   4/30 | MAE 0.4602 | best 0.4592 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:36:17 |    dc | Ridge regression          trial   5/30 | MAE 0.4624 | best 0.4592 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:36:18 |    dc | Ridge regression          trial   6/30 | MAE 0.4624 | best 0.4592 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:36:18 |    dc | Ridge regression          trial

,MAE,RMSE,R2,fit_s
model,,,,
Support vector regression,0.3792,0.6142,0.7851,203.1
"Weighted ensemble (Support vector regression 0.32, Extra trees 0.51, Hist gradient boosting 0.18)",0.3848,0.5460,0.8302,NaN
Extra trees,0.4054,0.5671,0.8168,122.7
Hist gradient boosting,0.4346,0.6121,0.7866,95.3
Random forest,0.4530,0.6333,0.7716,142.2
Ridge regression,0.4592,0.6611,0.7510,4.5
Baseline: multiple linear regression,0.4624,0.6650,0.7481,NaN
Baseline: 30-day moving average,0.8555,1.1588,0.2351,NaN
Baseline: historical average,0.9536,1.2522,0.1068,NaN


07:45:54 | >>> product group 2/4: Coconut Milk
07:45:54 |    cm | Ridge regression          trial   1/30 | MAE 2.7038 | best 2.7038 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:45:54 |    cm | Ridge regression          trial   2/30 | MAE 2.7255 | best 2.7038 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:45:54 |    cm | Ridge regression          trial   3/30 | MAE 2.6897 | best 2.6897 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:03
07:45:54 |    cm | Ridge regression          trial   4/30 | MAE 2.6929 | best 2.6897 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:45:55 |    cm | Ridge regression          trial   5/30 | MAE 2.7095 | best 2.6897 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:45:55 |    cm | Ridge regression          trial   6/30 | MAE 2.7095 | best 2.6897 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:45:55 |    cm | Ridge regression          trial   7/30 | MAE 2.7099 | best 2.6897 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:45:55 |    cm | Ridge regr

,MAE,RMSE,R2,fit_s
model,,,,
"Weighted ensemble (Ridge regression 0.73, Hist gradient boosting 0.18, Support vector regression 0.09)",2.6562,4.1458,0.9648,NaN
Ridge regression,2.6892,4.1601,0.9645,3.4
Baseline: multiple linear regression,2.7101,4.1839,0.9641,NaN
Hist gradient boosting,2.7782,4.3156,0.9618,115.3
Support vector regression,2.7784,4.3422,0.9614,64.3
Extra trees,2.9427,4.4714,0.9590,66.4
Random forest,2.9472,4.4429,0.9596,77.0
Baseline: 30-day moving average,4.0928,5.6983,0.9335,NaN
Baseline: historical average,4.4751,6.1892,0.9215,NaN


07:51:24 | >>> product group 3/4: VCO - Cold Press
07:51:24 |    vco_cold_press | Ridge regression trial   1/30 | MAE 0.6683 | best 0.6683 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:51:25 |    vco_cold_press | Ridge regression trial   2/30 | MAE 0.6596 | best 0.6596 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:51:25 |    vco_cold_press | Ridge regression trial   3/30 | MAE 0.6640 | best 0.6596 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:51:25 |    vco_cold_press | Ridge regression trial   4/30 | MAE 0.6667 | best 0.6596 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:02
07:51:25 |    vco_cold_press | Ridge regression trial   5/30 | MAE 0.6684 | best 0.6596 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:51:25 |    vco_cold_press | Ridge regression trial   6/30 | MAE 0.6684 | best 0.6596 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:51:25 |    vco_cold_press | Ridge regression trial   7/30 | MAE 0.6685 | best 0.6596 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:5

,MAE,RMSE,R2,fit_s
model,,,,
"Weighted ensemble (Ridge regression 0.61, Support vector regression 0.18, Hist gradient boosting 0.21)",0.6508,0.8303,0.8001,NaN
Ridge regression,0.6584,0.8366,0.7971,2.5
Support vector regression,0.6634,0.8426,0.7941,42.5
Baseline: multiple linear regression,0.6685,0.8462,0.7924,NaN
Hist gradient boosting,0.6866,0.8894,0.7707,136.1
Random forest,0.7100,0.9220,0.7535,57.1
Extra trees,0.7147,0.9176,0.7559,49.3
Baseline: historical average,1.4481,1.8588,-0.0017,NaN
Baseline: 30-day moving average,1.5359,2.0039,-0.1643,NaN


07:56:15 | >>> product group 4/4: VCO - Wet Centrifuge
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   1/30 | MAE 0.9000 | best 0.9000 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   2/30 | MAE 0.4935 | best 0.4935 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   3/30 | MAE 0.5536 | best 0.4935 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   4/30 | MAE 0.4672 | best 0.4672 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   5/30 | MAE 4.6179 | best 0.4672 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   6/30 | MAE 4.6183 | best 0.4672 |   0.1s/trial | elapsed 0:00:00 | ETA 0:00:01
07:56:15 |    vco_wet_centrifuge | Ridge regression trial   7/30 | MAE 5.9139 | best 0.4672 |   0.1s/trial | el

,MAE,RMSE,R2,fit_s
model,,,,
"Weighted ensemble (Ridge regression 0.54, Random forest 0.36, Hist gradient boosting 0.10)",0.4242,0.5337,0.6016,NaN
Ridge regression,0.4488,0.5654,0.5527,1.6
Extra trees,0.4615,0.5784,0.5320,13.7
Random forest,0.4641,0.5868,0.5183,16.5
Hist gradient boosting,0.4743,0.5971,0.5012,36.7
Support vector regression,0.5456,0.7080,0.2987,2.0
Baseline: 30-day moving average,0.5671,0.7044,0.3059,NaN
Baseline: historical average,0.6935,0.8693,-0.0572,NaN
Baseline: multiple linear regression,6.6765,12.8746,-230.8899,NaN


## 8. Final model: refit on 2016–2024, evaluate once on the unseen 2025 test set

In [9]:
FINAL, TEST_METRICS, CONFORMAL, SELECTED = {}, {}, {}, {}
for g in GROUPS:
    log(f"final refit on 2016-2024 and 2025 test evaluation: {g}")
    best_name = next(m for m in RESULTS[g].index if m in FITTED[g])
    SELECTED[g] = best_name
    _, Xtr, ytr = split_xy(g, ["train"])
    va_df, Xva, yva = split_xy(g, ["validation"])
    # split-conformal interval from validation residuals of the train-only fit
    res = np.abs(yva - FITTED[g][best_name].predict(Xva))
    CONFORMAL[g] = {"q80": float(np.quantile(res, 0.80)), "q90": float(np.quantile(res, 0.90))}
    trv_df, Xtrv, ytrv = split_xy(g, ["train", "validation"])
    final = clone(FITTED[g][best_name]).fit(Xtrv, ytrv)
    FINAL[g] = final
    te_df, Xte, yte = split_xy(g, ["test"])
    te_all, Xte_all, yte_all = split_xy(g, ["test"], include_downgraded=True)
    pred = final.predict(Xte)
    var_mean = trv_df.groupby("product_variant")[TARGET].mean()
    base_hist = te_df.product_variant.map(var_mean).fillna(ytrv.mean()).values
    base_ma = te_df.hist_yield_mean_30d_variant.fillna(te_df.product_variant.map(var_mean)).fillna(ytrv.mean()).values
    base_mlr = make_pipeline(g, LinearRegression(), True).fit(Xtrv, ytrv).predict(Xte)
    lo, hi = pred - CONFORMAL[g]["q90"], pred + CONFORMAL[g]["q90"]
    cat_true = [yield_level(g, v, y) for v, y in zip(te_df.product_variant, yte)]
    cat_pred = [yield_level(g, v, y) for v, y in zip(te_df.product_variant, pred)]
    TEST_METRICS[g] = {
        "selected_model": best_name,
        "test_n": int(len(yte)),
        "model": reg_metrics(yte, pred),
        "baseline_historical_average": reg_metrics(yte, base_hist),
        "baseline_moving_average": reg_metrics(yte, base_ma),
        "baseline_linear_regression": reg_metrics(yte, base_mlr),
        "model_incl_downgraded_batches": reg_metrics(yte_all, final.predict(Xte_all)),
        "interval90_coverage": float(np.mean((yte >= lo) & (yte <= hi))),
        "yield_level_macro_f1": float(f1_score(cat_true, cat_pred, average="macro")),
        "yield_level_accuracy": float(np.mean(np.array(cat_true) == np.array(cat_pred))),
    }
    m = TEST_METRICS[g]
    print(f"{g:22s} [{best_name}]  MAE {m['model']['MAE']:.3f} (hist {m['baseline_historical_average']['MAE']:.3f}, "
          f"MA {m['baseline_moving_average']['MAE']:.3f}, MLR {m['baseline_linear_regression']['MAE']:.3f})  "
          f"R2 {m['model']['R2']:.3f}  90%-interval coverage {m['interval90_coverage']:.2f}  level macro-F1 {m['yield_level_macro_f1']:.3f}")
summary = pd.DataFrame({g: {"selected": TEST_METRICS[g]["selected_model"], "MAE": TEST_METRICS[g]["model"]["MAE"],
                            "RMSE": TEST_METRICS[g]["model"]["RMSE"], "R2": TEST_METRICS[g]["model"]["R2"],
                            "MAE hist-avg": TEST_METRICS[g]["baseline_historical_average"]["MAE"],
                            "MAE moving-avg": TEST_METRICS[g]["baseline_moving_average"]["MAE"],
                            "MAE MLR": TEST_METRICS[g]["baseline_linear_regression"]["MAE"],
                            "beats all baselines": TEST_METRICS[g]["model"]["MAE"] < min(TEST_METRICS[g]["baseline_historical_average"]["MAE"],
                                                                                        TEST_METRICS[g]["baseline_moving_average"]["MAE"],
                                                                                        TEST_METRICS[g]["baseline_linear_regression"]["MAE"])}
                        for g in GROUPS}).T
display(summary)

07:57:27 | final refit on 2016-2024 and 2025 test evaluation: Desiccated Coconut
Desiccated Coconut     [Support vector regression]  MAE 0.406 (hist 0.959, MA 0.947, MLR 0.455)  R2 0.731  90%-interval coverage 0.88  level macro-F1 0.750
07:57:32 | final refit on 2016-2024 and 2025 test evaluation: Coconut Milk
Coconut Milk           [Weighted ensemble (Ridge regression 0.73, Hist gradient boosting 0.18, Support vector regression 0.09)]  MAE 2.998 (hist 6.136, MA 4.706, MLR 3.115)  R2 0.950  90%-interval coverage 0.87  level macro-F1 0.718
07:57:38 | final refit on 2016-2024 and 2025 test evaluation: VCO - Cold Press
VCO - Cold Press       [Weighted ensemble (Ridge regression 0.61, Support vector regression 0.18, Hist gradient boosting 0.21)]  MAE 0.661 (hist 1.674, MA 1.718, MLR 0.668)  R2 0.845  90%-interval coverage 0.93  level macro-F1 0.806
07:57:41 | final refit on 2016-2024 and 2025 test evaluation: VCO - Wet Centrifuge
VCO - Wet Centrifuge   [Weighted ensemble (Ridge regression 

,selected,MAE,RMSE,R2,MAE hist-avg,MAE moving-avg,MAE MLR,beats all baselines
Desiccated Coconut,Support vector regression,0.4063,0.7451,0.731,0.959,0.9467,0.4545,True
Coconut Milk,"Weighted ensemble (Ridge regression 0.73, Hist gradient boosting 0.18, Support vector regression 0.09)",2.9981,4.9329,0.9503,6.136,4.7058,3.1153,True
VCO - Cold Press,"Weighted ensemble (Ridge regression 0.61, Support vector regression 0.18, Hist gradient boosting 0.21)",0.6615,0.8242,0.8448,1.6743,1.7177,0.6683,True
VCO - Wet Centrifuge,"Weighted ensemble (Ridge regression 0.54, Random forest 0.36, Hist gradient boosting 0.10)",0.3925,0.4892,0.7786,0.8634,0.6155,0.4003,True


### Error analysis on the test set (SRS §5.6: by product, batch size, line and period)

In [10]:
for g in GROUPS:
    te_df, Xte, yte = split_xy(g, ["test"])
    e = te_df.assign(abs_error=np.abs(yte - FINAL[g].predict(Xte)), quarter=te_df.production_date.dt.quarter)
    print(f"\n{g}")
    display(pd.concat({k: e.groupby(k).abs_error.agg(["count", "mean"]).round(3)
                       for k in ["product_variant", "batch_size_category", "machine_id", "quarter"] if k in e}, axis=0))


Desiccated Coconut


count   mean
product_variant     Chips         105  0.336
                    Extra Fine     77  0.378
                    Fine          343  0.447
                    Medium        277  0.391
batch_size_category Large         228  0.458
                    Medium        253  0.353
                    Small         321  0.411
machine_id          DRY-A         480  0.432
                    DRY-B         322  0.368
quarter             1             120  0.529
                    2             186  0.323
                    3             270  0.339
                    4             226  0.490


Coconut Milk


count   mean
product_variant     Coconut Cream 24%    162  2.582
                    Coconut Milk 17%     299  3.342
                    Coconut Milk 20%     149  2.759
batch_size_category Large                184  2.956
                    Medium               191  3.107
                    Small                235  2.943
machine_id          PRS-1                378  2.903
                    PRS-2                232  3.153
quarter             1                    108  2.928
                    2                    156  3.402
                    3                    181  2.853
                    4                    165  2.821


VCO - Cold Press


count   mean
product_variant     Virgin Coconut Oil    450  0.662
batch_size_category Large                 125  0.700
                    Medium                125  0.612
                    Small                 200  0.668
machine_id          EXP-1                 305  0.654
                    EXP-2                 145  0.677
quarter             1                     113  0.612
                    2                     108  0.640
                    3                     113  0.731
                    4                     116  0.662


VCO - Wet Centrifuge


count   mean
product_variant     Virgin Coconut Oil    191  0.392
batch_size_category Large                  48  0.410
                    Medium                 53  0.393
                    Small                  90  0.383
machine_id          CFG-1                 191  0.392
quarter             1                      42  0.376
                    2                      45  0.352
                    3                      54  0.417
                    4                      50  0.416

### Most influential inputs (permutation importance on the test set)

In [11]:
IMPORTANCE = {}
for g in GROUPS:
    te_df, Xte, yte = split_xy(g, ["test"])
    n = min(len(Xte), 400 if FAST_MODE else 1500)
    pi = permutation_importance(FINAL[g], Xte.iloc[:n], yte[:n], n_repeats=2 if FAST_MODE else 5,
                                random_state=RANDOM_STATE, scoring="neg_mean_absolute_error")
    imp = pd.Series(pi.importances_mean, index=Xte.columns).sort_values(ascending=False)
    IMPORTANCE[g] = imp
    print(f"\n{g}: top inputs (increase in MAE when shuffled)")
    display(imp.head(12).round(4).to_frame("MAE increase"))


Desiccated Coconut: top inputs (increase in MAE when shuffled)


,MAE increase
lab_kernel_moisture_pct,0.5413
product_variant,0.1939
product_specific_feed_kg,0.0842
planned_dryer_inlet_temp_c,0.0443
lab_kernel_fat_pct_wb,0.0384
planned_dryer_residence_min,0.0223
blanching_method,0.0181
ambient_rh_pct,0.0132
hist_yield_mean_90d_variant,0.0131
hist_yield_mean_30d_variant,0.0107



Coconut Milk: top inputs (increase in MAE when shuffled)


,MAE increase
target_fat_pct,10.9841
product_variant,1.7746
planned_water_kernel_ratio,1.7447
lab_kernel_fat_pct_wb,1.5576
immature_nut_pct,0.8296
hist_yield_mean_90d_variant,0.6221
days_since_maintenance,0.6211
lab_kernel_moisture_pct,0.3275
extraction_water_temp_c,0.2633
hist_yield_mean_30d_variant,0.1675



VCO - Cold Press: top inputs (increase in MAE when shuffled)


,MAE increase
planned_expeller_passes,0.5891
lab_kernel_fat_pct_wb,0.5606
days_since_maintenance,0.3162
machine_id,0.2710
feed_moisture_pct,0.2298
lab_kernel_moisture_pct,0.1523
immature_nut_pct,0.0900
machine_age_years,0.0258
planned_feed_rate_kg_h,0.0231
retail_nut_price_lkr_lag1m,0.0045



VCO - Wet Centrifuge: top inputs (increase in MAE when shuffled)


,MAE increase
lab_kernel_fat_pct_wb,0.3204
days_since_maintenance,0.0757
lab_kernel_moisture_pct,0.0725
immature_nut_pct,0.0683
hist_yield_mean_90d_variant,0.0081
crew_team,0.0028
ambient_rh_pct,0.0020
national_coconut_prod_mn_nuts_lag2m,0.0017
kernel_to_dryer_delay_h,0.0015
event_context,0.0014


## 9. Save model packages (SRS §3.3 artifact contract)

In [12]:
def build_schema(g):
    tr = df[(df.model_group == g) & (df.split.isin(["train", "validation"]))]
    recent = df[(df.model_group == g) & (df.production_date > df.production_date.max() - pd.Timedelta(days=365))]
    fields = []
    for c in FEATURES[g]["raw"]:
        info = dd[dd.field == c].iloc[0] if (dd.field == c).any() else None
        entry = {"name": c, "required": c in FEATURES[g]["required"], "unit": None if info is None else str(info.unit),
                 "description": None if info is None else str(info.description)}
        if c in FEATURES[g]["cat"]:
            vc = tr[c].dropna().astype(str)
            rc = recent[c].dropna().astype(str)
            entry.update({"type": "categorical", "categories": sorted(vc.unique().tolist()),
                          "default": str((rc if len(rc) else vc).mode().iloc[0])})
        else:
            s = pd.to_numeric(tr[c], errors="coerce").dropna()
            rs_ = pd.to_numeric(recent[c], errors="coerce").dropna()
            entry.update({"type": "numeric", "default": float((rs_ if len(rs_) else s).median()), "min": float(s.min()), "max": float(s.max()),
                          "p01": float(s.quantile(0.01)), "p99": float(s.quantile(0.99))})
        fields.append(entry)
    return {"component": "yield_prediction", "model_group": g, "target": TARGET, "prediction_point": "after preprocessing, before main processing",
            "inputs": fields, "engineered_features": FEATURES[g]["engineered"], "feature_module": "c1_features.py"}


for g, s in GROUPS.items():
    d = ART / s
    d.mkdir(exist_ok=True)
    joblib.dump(FINAL[g], d / "model_pipeline.joblib")
    json.dump(build_schema(g), open(d / "feature_schema.json", "w"), indent=2)
    meta = {"component": "yield_prediction", "model_group": g, "model_version": f"c1-{s}-v1.0", "selected_model": SELECTED[g],
            "hyperparameters": {k: str(v) for k, v in BEST_PARAMS[g].get(SELECTED[g], {}).items()},
            "target": TARGET, "training_period": "2016-01-01..2024-12-31 (final refit)", "selection_period": "2023-01-01..2024-12-31",
            "test_period": "2025-01-01..2025-12-31", "threshold_version": THRESHOLDS["version"], "conformal_abs_residual": CONFORMAL[g],
            "excluded_from_training": "QC status Rejected/Downgraded", "created_at": pd.Timestamp.now().isoformat(timespec="seconds"),
            "library_versions": LIB_VERSIONS, "fast_mode": FAST_MODE}
    json.dump(meta, open(d / "metadata.json", "w"), indent=2)
    json.dump({"validation_leaderboard": RESULTS[g].reset_index().to_dict(orient="records"), "test": TEST_METRICS[g],
               "top_inputs": IMPORTANCE[g].head(15).round(5).to_dict()}, open(d / "metrics.json", "w"), indent=2, default=float)
json.dump(THRESHOLDS, open(ART / "yield_thresholds_v1.json", "w"), indent=2)
card = ["# Component 1 – Batch Yield models (model card)", "",
        "Intended use: decision support for production/quality officers before a batch is processed. Not for automatic decisions.", "",
        "| Model group | Selected model | Test MAE | Test R² | Hist-avg MAE | MLR MAE | 90 % interval ± |", "|---|---|---|---|---|---|---|"]
for g in GROUPS:
    m = TEST_METRICS[g]
    card.append(f"| {g} | {SELECTED[g]} | {m['model']['MAE']:.3f} | {m['model']['R2']:.3f} | {m['baseline_historical_average']['MAE']:.3f} | "
                f"{m['baseline_linear_regression']['MAE']:.3f} | {CONFORMAL[g]['q90']:.2f} |")
card += ["", "Limitations: trained on 2016–2025 batch records of one factory; predictions outside the input ranges in feature_schema.json "
         "raise warnings; whole-batch microbiological rejections are not predicted (handled by QC).", "",
         "Files per group: model_pipeline.joblib, feature_schema.json, metadata.json, metrics.json. Shared: c1_features.py, yield_thresholds_v1.json."]
(ART / "MODEL_CARD.md").write_text("\n".join(card), encoding="utf-8")
print("Saved to", ART)
for p in sorted(ART.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(ART))

Saved to E:\Coco Optima\ml-components\01-yield-prediction\model_artifacts
   c1_features.py
   cm\feature_schema.json
   cm\metadata.json
   cm\metrics.json
   cm\model_pipeline.joblib
   dc\feature_schema.json
   dc\metadata.json
   dc\metrics.json
   dc\model_pipeline.joblib
   MODEL_CARD.md
   training_progress.log
   vco_cold_press\feature_schema.json
   vco_cold_press\metadata.json
   vco_cold_press\metrics.json
   vco_cold_press\model_pipeline.joblib
   vco_wet_centrifuge\feature_schema.json
   vco_wet_centrifuge\metadata.json
   vco_wet_centrifuge\metrics.json
   vco_wet_centrifuge\model_pipeline.joblib
   yield_thresholds_v1.json


## 10. Manual testing – enter values and get a prediction
The functions below load the **saved** packages from `model_artifacts/` (exactly what the backend will use).

* `predict_yield(inputs)` – validates inputs (required fields, units/ranges), routes to the right product model, and returns
  predicted saleable yield %, a 90 % interval, expected accepted output, the High/Moderate/Low level, the main factors and warnings.
* Missing **required** fields → no prediction (SRS YLD-NFR-05). Missing optional fields → filled with typical values and listed in the warnings.

In [13]:
import c1_features

ARTS = {}
for g, s in GROUPS.items():
    d = ART / s
    ARTS[g] = {"model": joblib.load(d / "model_pipeline.joblib"), "schema": json.load(open(d / "feature_schema.json")),
               "meta": json.load(open(d / "metadata.json"))}
THR = json.load(open(ART / "yield_thresholds_v1.json"))["thresholds"]
HARD_LIMITS = {"product_specific_feed_kg": (1, 20000), "lab_kernel_moisture_pct": (20, 75), "lab_kernel_fat_pct_wb": (10, 60),
               "feed_moisture_pct": (0.5, 30), "planned_dryer_inlet_temp_c": (40, 140), "target_fat_pct": (5, 40),
               "planned_water_kernel_ratio": (0, 3), "immature_nut_pct": (0, 100), "germinated_nut_pct": (0, 100)}


def route_group(inputs):
    p = str(inputs.get("product_code", inputs.get("product_name", ""))).strip().lower()
    if p in ("dc", "desiccated coconut"):
        return "Desiccated Coconut"
    if p in ("cm", "coconut milk"):
        return "Coconut Milk"
    if p in ("vco", "virgin coconut oil"):
        return "VCO - Wet Centrifuge" if "centrifuge" in str(inputs.get("extraction_method", "")).lower() else "VCO - Cold Press"
    raise ValueError("product_code must be DC, CM or VCO")


def yield_level_for(g, variant, y):
    key = f"{g} | {variant}"
    if key not in THR:
        key = next(k for k in THR if k.startswith(g))
    t = THR[key]
    return "Low" if y < t["low_below"] else ("High" if y >= t["high_at_or_above"] else "Moderate")


def predict_yield(inputs, explain=True, show=True):
    try:
        g = route_group(inputs)
    except ValueError as e:
        return {"success": False, "errors": [str(e)]}
    art = ARTS[g]
    schema = {f["name"]: f for f in art["schema"]["inputs"]}
    errors, warns, defaulted = [], [], []
    for f in art["schema"]["inputs"]:
        if f["required"] and (inputs.get(f["name"]) is None or str(inputs.get(f["name"])).strip() == ""):
            errors.append(f"missing required field: {f['name']}")
    if not inputs.get("production_date"):
        errors.append("missing required field: production_date")
    row = {}
    for name, f in schema.items():
        v = inputs.get(name)
        if v is None or (isinstance(v, str) and v.strip() == ""):
            row[name] = f["default"]
            defaulted.append(name)
            continue
        if f["type"] == "numeric":
            try:
                v = float(v)
            except (TypeError, ValueError):
                errors.append(f"{name} must be numeric (got {v!r})")
                continue
            lo, hi = HARD_LIMITS.get(name, (None, None))
            if lo is not None and not (lo <= v <= hi):
                errors.append(f"{name}={v} outside physical limits [{lo}, {hi}]")
            elif not (f["p01"] <= v <= f["p99"]):
                warns.append(f"{name}={v} is outside the usual range [{f['p01']:.2f}, {f['p99']:.2f}] - prediction less reliable")
        else:
            v = str(v)
            if v not in f["categories"]:
                warns.append(f"{name}='{v}' not seen in training (known: {f['categories'][:8]})")
        row[name] = v
    if errors:
        return {"success": False, "model_group": g, "errors": errors}
    try:
        date = pd.Timestamp(inputs["production_date"])
        for k, v in {"month": date.month, "quarter": date.quarter, "day_of_week": date.dayofweek}.items():
            if k in schema and k in defaulted:
                row[k] = v
                defaulted.remove(k)
        if "monsoon_season" in schema and "monsoon_season" in defaulted:
            row["monsoon_season"] = "SW monsoon" if date.month in (5, 6, 7, 8, 9) else "NE monsoon" if date.month in (12, 1, 2) else "Inter-monsoon"
            defaulted.remove("monsoon_season")
    except Exception:
        return {"success": False, "model_group": g, "errors": ["production_date is not a valid date"]}
    X = pd.DataFrame([row])
    model = art["model"]
    pred = float(model.predict(X)[0])
    q = art["meta"]["conformal_abs_residual"]
    feed = float(row["product_specific_feed_kg"])
    variant = row.get("product_variant", "Virgin Coconut Oil")
    out = {"success": True, "model_group": g, "model_version": art["meta"]["model_version"],
           "predicted_yield_percentage": round(pred, 3),
           "interval_80": [round(pred - q["q80"], 3), round(pred + q["q80"], 3)],
           "interval_90": [round(pred - q["q90"], 3), round(pred + q["q90"], 3)],
           "expected_accepted_output_kg": round(pred * feed / 100, 1),
           "yield_level": yield_level_for(g, variant, pred), "threshold_version": "yield-thresholds-v1",
           "auto_filled_fields": defaulted,
           "warnings": warns + ([f"{len(defaulted)} optional/context fields auto-filled with recent typical values "
                                 f"(see 'auto_filled_fields')"] if defaulted else [])}
    if explain:
        cols = [c for c in schema if c not in defaulted]
        alt = pd.DataFrame([row] * len(cols))
        for i, c in enumerate(cols):
            alt.iloc[i, alt.columns.get_loc(c)] = schema[c]["default"]
        delta = pred - model.predict(alt)
        top = sorted(zip(cols, delta), key=lambda t: -abs(t[1]))[:6]
        out["top_factors"] = [{"input": c, "value": row[c], "typical": schema[c]["default"], "effect_on_yield_pts": round(float(dv), 3)} for c, dv in top]
    if show:
        print(f"Model: {g}  ({out['model_version']})")
        print(f"Predicted saleable yield : {out['predicted_yield_percentage']:.2f} %   90% interval {out['interval_90']}")
        print(f"Expected accepted output : {out['expected_accepted_output_kg']:,.1f} kg  (feed {feed:,.0f} kg)")
        print(f"Yield level              : {out['yield_level']}")
        if explain:
            print("Main factors (vs. typical value):")
            for t in out["top_factors"]:
                print(f"   {t['input']:34s} value={t['value']!s:>14}  typical={t['typical']!s:>10}  effect={t['effect_on_yield_pts']:+.3f} pts")
        for w in out["warnings"]:
            print("WARNING:", w)
    return out


print("Loaded models:", {g: a["meta"]["selected_model"] for g, a in ARTS.items()})

Loaded models: {'Desiccated Coconut': 'Support vector regression', 'Coconut Milk': 'Weighted ensemble (Ridge regression 0.73, Hist gradient boosting 0.18, Support vector regression 0.09)', 'VCO - Cold Press': 'Weighted ensemble (Ridge regression 0.61, Support vector regression 0.18, Hist gradient boosting 0.21)', 'VCO - Wet Centrifuge': 'Weighted ensemble (Ridge regression 0.54, Random forest 0.36, Hist gradient boosting 0.10)'}


### 10.1 Check with real 2025 test batches (never used for training)

In [14]:
check = []
for g in GROUPS:
    te = df[(df.model_group == g) & (df.split == "test") & (df.qc_status != "Rejected/Downgraded")].sample(3, random_state=7)
    for _, r in te.iterrows():
        inp = {c: (None if pd.isna(r[c]) else r[c]) for c in FEATURES[g]["raw"]}
        inp.update({"product_code": r.product_code, "extraction_method": r.extraction_method, "production_date": str(r.production_date.date())})
        o = predict_yield(inp, explain=False, show=False)
        check.append({"batch_id": r.batch_id, "model_group": g, "actual_yield_%": round(r[TARGET], 3),
                      "predicted_yield_%": o["predicted_yield_percentage"], "interval_90": o["interval_90"],
                      "actual_output_kg": round(r.accepted_output_kg, 1), "predicted_output_kg": o["expected_accepted_output_kg"]})
display(pd.DataFrame(check))

,batch_id,model_group,actual_yield_%,predicted_yield_%,interval_90,actual_output_kg,predicted_output_kg
0,DC-20250210-01,Desiccated Coconut,44.726,45.917,"[45.173, 46.66]",1198.3,1230.2
1,DC-20251009-02,Desiccated Coconut,46.618,46.837,"[46.093, 47.58]",1370.2,1376.6
2,DC-20250906-01,Desiccated Coconut,47.914,48.084,"[47.341, 48.828]",1521.8,1527.2
3,CM-20250916-01,Coconut Milk,118.920,123.677,"[118.558, 128.796]",2889.4,3005.0
4,CM-20250715-04,Coconut Milk,137.961,133.051,"[127.932, 138.169]",2606.5,2513.7
5,CM-20250811-01,Coconut Milk,154.427,154.319,"[149.2, 159.437]",4901.5,4898.1
6,VCO-20251219-01,VCO - Cold Press,47.508,48.747,"[47.352, 50.142]",233.5,239.6
7,VCO-20250927-02,VCO - Cold Press,50.086,49.345,"[47.95, 50.739]",408.7,402.7
8,VCO-20250619-02,VCO - Cold Press,50.477,49.671,"[48.276, 51.066]",291.2,286.6
9,VCF-20250215-02,VCO - Wet Centrifuge,18.889,19.183,"[18.34, 20.026]",105.8,107.4


### 10.2 ✏️ Enter your own values – Desiccated Coconut
Edit the numbers/text below and run the cell. Only the **required** fields are mandatory; you may add any other input listed in
`model_artifacts/dc/feature_schema.json` (for example `immature_nut_pct`, `days_since_maintenance`, `ambient_rh_pct`).

In [15]:
dc_input = {
    "product_code": "DC",
    "production_date": "2025-11-20",
    "product_variant": "Fine",                 # Fine | Medium | Extra Fine | Chips
    "machine_id": "DRY-B",                     # DRY-A | DRY-B
    "line_id": "DC-L2",
    "shift": "Day",                            # Day | Night
    "crew_team": "A",                          # A | B | C
    "product_specific_feed_kg": 3100,          # pared white kernel, kg
    "lab_kernel_moisture_pct": 48.4,
    "lab_kernel_fat_pct_wb": 34.0,
    "immature_nut_pct": 4.5,
    "germinated_nut_pct": 1.2,
    "nut_storage_days": 14,
    "paring_loss_pct": 13.5,
    "blanching_method": "Steam",
    "planned_dryer_inlet_temp_c": 91.0,
    "planned_dryer_residence_min": 56,
    "days_since_maintenance": 12,
    "ambient_rh_pct": 82,
}
result = predict_yield(dc_input)

Model: Desiccated Coconut  (c1-dc-v1.0)
Predicted saleable yield : 47.89 %   90% interval [47.145, 48.633]
Expected accepted output : 1,484.6 kg  (feed 3,100 kg)
Yield level              : Moderate
Main factors (vs. typical value):
   planned_dryer_inlet_temp_c         value=          91.0  typical=      95.4  effect=+0.223 pts
   lab_kernel_moisture_pct            value=          48.4  typical=     48.57  effect=+0.118 pts
   product_specific_feed_kg           value=        3100.0  typical=   2775.95  effect=+0.095 pts
   crew_team                          value=             A  typical=         B  effect=+0.078 pts
   line_id                            value=         DC-L2  typical=     DC-L1  effect=-0.066 pts
   machine_id                         value=         DRY-B  typical=     DRY-A  effect=-0.055 pts


### 10.3 ✏️ Enter your own values – Coconut Milk / Cream

In [16]:
cm_input = {
    "product_code": "CM",
    "production_date": "2025-11-20",
    "product_variant": "Coconut Milk 17%",     # Coconut Milk 17% | Coconut Milk 20% | Coconut Cream 24%
    "target_fat_pct": 17,
    "planned_water_kernel_ratio": 0.62,
    "extraction_water_temp_c": 42,
    "pack_format": "Can 400 ml",               # Can 400 ml | UHT Tetra 1 L | Bulk aseptic 20 L
    "machine_id": "PRS-1",
    "shift": "Day",
    "crew_team": "B",
    "product_specific_feed_kg": 2200,
    "lab_kernel_moisture_pct": 48.0,
    "lab_kernel_fat_pct_wb": 34.2,
    "immature_nut_pct": 5.0,
    "days_since_maintenance": 10,
}
result = predict_yield(cm_input)

Model: Coconut Milk  (c1-cm-v1.0)
Predicted saleable yield : 146.39 %   90% interval [141.273, 151.51]
Expected accepted output : 3,220.6 kg  (feed 2,200 kg)
Yield level              : Moderate
Main factors (vs. typical value):
   target_fat_pct                     value=          17.0  typical=      20.0  effect=+14.683 pts
   planned_water_kernel_ratio         value=          0.62  typical=      0.52  effect=+2.045 pts
   lab_kernel_fat_pct_wb              value=          34.2  typical=     33.74  effect=+1.087 pts
   lab_kernel_moisture_pct            value=          48.0  typical=     48.57  effect=+0.499 pts
   days_since_maintenance             value=          10.0  typical=      13.0  effect=+0.441 pts
   immature_nut_pct                   value=           5.0  typical=      4.24  effect=-0.349 pts


### 10.4 ✏️ Enter your own values – Virgin Coconut Oil (cold press or wet centrifuge)

In [17]:
vco_input = {
    "product_code": "VCO",
    "extraction_method": "Cold Press",         # Cold Press | Wet Centrifuge
    "production_date": "2025-11-20",
    "machine_id": "EXP-2",                     # EXP-1 | EXP-2  (CFG-1 for wet centrifuge)
    "shift": "Day",
    "crew_team": "A",
    "product_specific_feed_kg": 720,           # dried kernel (cold press) or fresh kernel (centrifuge), kg
    "lab_kernel_moisture_pct": 48.0,
    "lab_kernel_fat_pct_wb": 34.5,
    "feed_moisture_pct": 7.0,                  # dried-kernel moisture before expelling
    "planned_feed_rate_kg_h": 42,
    "planned_expeller_passes": 2,
    "kernel_to_dryer_delay_h": 1.0,
    "days_since_maintenance": 8,
}
result = predict_yield(vco_input)

Model: VCO - Cold Press  (c1-vco_cold_press-v1.0)
Predicted saleable yield : 49.47 %   90% interval [48.077, 50.866]
Expected accepted output : 356.2 kg  (feed 720 kg)
Yield level              : High
Main factors (vs. typical value):
   machine_id                         value=         EXP-2  typical=     EXP-1  effect=+1.177 pts
   lab_kernel_fat_pct_wb              value=          34.5  typical=     33.56  effect=+0.618 pts
   lab_kernel_moisture_pct            value=          48.0  typical=      48.8  effect=-0.314 pts
   days_since_maintenance             value=           8.0  typical=      10.0  effect=+0.201 pts
   monsoon_season                     value= Inter-monsoon  typical=SW monsoon  effect=+0.030 pts
   product_specific_feed_kg           value=         720.0  typical=     627.6  effect=+0.026 pts


### 10.5 What-if analysis – how does one input change the predicted yield?

In [18]:
def what_if(inputs, field, values):
    rows = []
    for v in values:
        o = predict_yield({**inputs, field: v}, explain=False, show=False)
        rows.append({field: v, "predicted_yield_%": o.get("predicted_yield_percentage"),
                     "expected_output_kg": o.get("expected_accepted_output_kg"), "level": o.get("yield_level"),
                     "note": "; ".join(o.get("errors", [])) if not o["success"] else ""})
    return pd.DataFrame(rows)


display(what_if(dc_input, "planned_dryer_inlet_temp_c", [86, 88, 90, 92, 94, 96, 98, 100, 102]))
display(what_if(cm_input, "planned_water_kernel_ratio", [0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75]))
display(what_if(vco_input, "feed_moisture_pct", [4, 5, 6, 7, 8, 9, 10, 11]))

,planned_dryer_inlet_temp_c,predicted_yield_%,expected_output_kg,level,note
0,86,48.105,1491.2,Moderate,
1,88,48.024,1488.7,Moderate,
2,90,47.936,1486.0,Moderate,
3,92,47.841,1483.1,Moderate,
4,94,47.740,1479.9,Moderate,
5,96,47.633,1476.6,Moderate,
6,98,47.522,1473.2,Moderate,
7,100,47.406,1469.6,Moderate,
8,102,47.286,1465.9,Moderate,


,planned_water_kernel_ratio,predicted_yield_%,expected_output_kg,level,note
0,0.45,142.982,3145.6,Low,
1,0.50,144.034,3168.7,Low,
2,0.55,144.840,3186.5,Moderate,
3,0.60,146.053,3213.2,Moderate,
4,0.65,147.014,3234.3,Moderate,
5,0.70,148.078,3257.7,Moderate,
6,0.75,148.920,3276.3,Moderate,


,feed_moisture_pct,predicted_yield_%,expected_output_kg,level,note
0,4,48.345,348.1,Moderate,
1,5,49.190,354.2,High,
2,6,49.587,357.0,High,
3,7,49.471,356.2,High,
4,8,48.885,352.0,High,
5,9,47.833,344.4,Moderate,
6,10,46.258,333.1,Moderate,
7,11,44.615,321.2,Low,


### 10.6 Input validation examples (no fabricated result when inputs are invalid – SRS YLD-NFR-05)

In [19]:
print(predict_yield({"product_code": "DC", "production_date": "2025-11-20", "machine_id": "DRY-B"}, show=False))
print(predict_yield({**dc_input, "lab_kernel_moisture_pct": 95}, show=False))
print(predict_yield({**dc_input, "product_code": "XYZ"}, show=False))

{'success': False, 'model_group': 'Desiccated Coconut', 'errors': ['missing required field: product_variant', 'missing required field: shift', 'missing required field: product_specific_feed_kg', 'missing required field: lab_kernel_moisture_pct', 'missing required field: planned_dryer_inlet_temp_c', 'missing required field: planned_dryer_residence_min']}
{'success': False, 'model_group': 'Desiccated Coconut', 'errors': ['lab_kernel_moisture_pct=95.0 outside physical limits [20, 75]']}
{'success': False, 'errors': ['product_code must be DC, CM or VCO']}


### 10.7 Batch predictions from a CSV file

In [20]:
def predict_csv(path_in, path_out):
    inp = pd.read_csv(path_in)
    outs = [predict_yield({k: (None if pd.isna(v) else v) for k, v in r.items()}, explain=False, show=False) for _, r in inp.iterrows()]
    res = inp.assign(predicted_yield_pct=[o.get("predicted_yield_percentage") for o in outs],
                     expected_accepted_output_kg=[o.get("expected_accepted_output_kg") for o in outs],
                     yield_level=[o.get("yield_level") for o in outs],
                     messages=["; ".join(o.get("errors", []) + o.get("warnings", [])) for o in outs])
    res.to_csv(path_out, index=False)
    return res


demo = df[df.split == "test"].sample(6, random_state=1)
demo_in = demo[["product_code", "extraction_method", "production_date", "product_variant", "machine_id", "shift", "crew_team",
                "product_specific_feed_kg", "lab_kernel_moisture_pct", "lab_kernel_fat_pct_wb", "target_fat_pct",
                "planned_water_kernel_ratio", "planned_dryer_inlet_temp_c", "planned_dryer_residence_min", "feed_moisture_pct",
                "planned_feed_rate_kg_h", "planned_expeller_passes"]].assign(production_date=lambda d: d.production_date.dt.date.astype(str))
demo_in.to_csv(ART / "manual_test_input_example.csv", index=False)
display(predict_csv(ART / "manual_test_input_example.csv", ART / "manual_test_predictions.csv"))

,product_code,extraction_method,production_date,product_variant,machine_id,shift,crew_team,product_specific_feed_kg,lab_kernel_moisture_pct,lab_kernel_fat_pct_wb,target_fat_pct,planned_water_kernel_ratio,planned_dryer_inlet_temp_c,planned_dryer_residence_min,feed_moisture_pct,planned_feed_rate_kg_h,planned_expeller_passes,predicted_yield_pct,expected_accepted_output_kg,yield_level,messages
0,DC,NaN,2025-12-12,Fine,DRY-B,Night,A,2770.1,49.38,33.53,NaN,NaN,89.8,61.0,NaN,NaN,NaN,46.982,1301.4,Moderate,24 optional/context fields auto-filled with recent typical values (see 'auto_filled_fields')
1,DC,NaN,2025-01-14,Medium,DRY-B,Day,A,2998.1,50.06,31.51,NaN,NaN,93.1,60.0,NaN,NaN,NaN,46.027,1379.9,Low,"lab_kernel_fat_pct_wb=31.51 is outside the usual range [31.59, 36.26] - prediction less reliable; 24 optional/context fields auto-filled with recent typical values (see 'auto_filled_fields')"
2,CM,Two-stage screw press,2025-05-21,Coconut Milk 17%,PRS-1,Day,A,2068.9,48.42,33.90,17.0,0.61,NaN,NaN,NaN,NaN,NaN,144.798,2995.7,Moderate,25 optional/context fields auto-filled with recent typical values (see 'auto_filled_fields')
3,VCO,Cold Press,2025-05-07,Virgin Coconut Oil,EXP-2,Day,B,848.7,48.94,33.19,NaN,NaN,57.1,510.0,7.67,45.0,1.0,45.031,382.2,Low,24 optional/context fields auto-filled with recent typical values (see 'auto_filled_fields')
4,DC,NaN,2025-11-10,Chips,DRY-A,Night,C,2615.1,48.59,33.53,NaN,NaN,98.2,64.0,NaN,NaN,NaN,47.264,1236.0,Moderate,24 optional/context fields auto-filled with recent typical values (see 'auto_filled_fields')
5,DC,NaN,2025-04-03,Extra Fine,DRY-B,Day,C,2816.7,48.28,33.37,NaN,NaN,92.8,57.0,NaN,NaN,NaN,46.577,1311.9,Moderate,24 optional/context fields auto-filled with recent typical values (see 'auto_filled_fields')


### 10.8 Optional: interactive form (requires `ipywidgets`)
Choose a product, fill the boxes and click **Predict**. Fields left empty use typical values.

In [21]:
if widgets is None:
    print("ipywidgets is not installed - use the dictionaries in 10.2-10.4 instead (pip install ipywidgets).")
else:
    from IPython.display import clear_output
    product_box = widgets.Dropdown(options=list(GROUPS), description="Model", style={"description_width": "160px"})
    form_area, out_area = widgets.VBox(), widgets.Output()
    form_widgets = {}

    def build_form(change=None):
        g = product_box.value
        form_widgets.clear()
        items = [widgets.Text(value="2025-11-20", description="production_date", style={"description_width": "260px"})]
        form_widgets["production_date"] = items[0]
        for f in ARTS[g]["schema"]["inputs"]:
            label = f["name"] + (" *" if f["required"] else "")
            if f["type"] == "categorical":
                w = widgets.Dropdown(options=[""] + f["categories"], value=f["default"] if f["required"] else "", description=label,
                                     style={"description_width": "260px"})
            else:
                w = widgets.Text(value=str(round(f["default"], 3)) if f["required"] else "", placeholder=f"typical {f['default']:.3g}",
                                 description=label, style={"description_width": "260px"})
            form_widgets[f["name"]] = w
            items.append(w)
        form_area.children = items

    def on_predict(_):
        g = product_box.value
        code_map = {"Desiccated Coconut": ("DC", ""), "Coconut Milk": ("CM", ""), "VCO - Cold Press": ("VCO", "Cold Press"),
                    "VCO - Wet Centrifuge": ("VCO", "Wet Centrifuge")}
        vals = {k: (w.value if w.value != "" else None) for k, w in form_widgets.items()}
        vals["product_code"], method = code_map[g]
        if method:
            vals["extraction_method"] = method
        with out_area:
            clear_output()
            o = predict_yield(vals)
            if not o["success"]:
                print("NO PREDICTION:", *o["errors"], sep="\n - ")

    product_box.observe(build_form, names="value")
    button = widgets.Button(description="Predict", button_style="success")
    button.on_click(on_predict)
    build_form()
    display(widgets.VBox([product_box, form_area, button, out_area]))

ipywidgets is not installed - use the dictionaries in 10.2-10.4 instead (pip install ipywidgets).
